# 09 - Advanced RAG


## What this notebook is for

This notebook is written as **notes + code**. Read the Markdown cells first, then run the code cells.

The goal is to understand the concept, see the architecture, implement a small example, and then evaluate/improve it.

## Learning objectives

- Understand advanced retrieval pipelines.
- Implement query rewriting, hybrid retrieval, reranking, compression, and citations conceptually.
- Evaluate each stage separately.
- Learn a debugging order for poor RAG.

## Core AI concept

Advanced RAG is not one feature. It is a collection of retrieval and context-engineering techniques. The right technique depends on the failure mode: poor recall, noisy ranking, missing query intent, too much context, or unsupported generation.

## Architecture / mental model

```text
Question
 ↓
Query transform
 ↓
Hybrid / semantic retrieval
 ↓
Metadata filters
 ↓
Reranker
 ↓
Context compression
 ↓
LLM + citations
```

## Connection to the supplied learning material

Your RAG notebook includes reranking with a relevance score and a top-3 context selection step. This notebook expands that into a systematic advanced RAG architecture.

### Source foundation
This notebook is an expanded learning version of the uploaded notebooks. The source material for this track includes:
- `chunking.ipynb`
- `embeddings.ipynb`
- `faiss.ipynb`
- `chroma_db.ipynb`
- `searching.ipynb`
- `rag.ipynb`

The original examples and terminology are retained where useful, then organized into a cleaner learning sequence.

## Setup

The code below uses environment variables for secrets and model names so the notebook is easier to maintain when providers or model versions change.

In [ ]:
from pathlib import Path
import os
from dotenv import load_dotenv

load_dotenv()

OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-5.6-luna")
OPENAI_EMBED_MODEL = os.getenv("OPENAI_EMBED_MODEL", "text-embedding-3-small")

print("Model:", OPENAI_MODEL)
print("Embedding model:", OPENAI_EMBED_MODEL)

In [ ]:
%pip install -qU langchain langchain-openai langchain-chroma langchain-text-splitters

### Query rewriting

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(model=OPENAI_MODEL, temperature=0)

rewrite_prompt = ChatPromptTemplate.from_template(
    "Rewrite this user query into one precise search query.\nUser query: {query}"
)

def rewrite_query(query):
    return (rewrite_prompt | llm | StrOutputParser()).invoke({"query": query})

# print(rewrite_query("Tell me stuff about agents using tools"))

### Reranking skeleton

In [ ]:
def rerank(question, candidate_docs, scorer):
    scored = []
    for doc in candidate_docs:
        score = scorer(question, doc.page_content)
        scored.append((score, doc))
    scored.sort(key=lambda x: x[0], reverse=True)
    return [doc for score, doc in scored]

# In production, scorer can be a cross-encoder or another relevance model.

### Context compression

In [ ]:
def compress_context(question, docs, keep_chars=500):
    # Educational placeholder. A real compressor should be relevance-aware.
    compressed = []
    for d in docs:
        compressed.append(
            f"[{d.metadata.get('source', 'unknown')}] "
            + d.page_content[:keep_chars]
        )
    return "\n\n".join(compressed)

### Experiment matrix

In [ ]:
experiments = [
    {"chunk_size": 250, "k": 3, "rerank": False},
    {"chunk_size": 500, "k": 5, "rerank": True},
    {"chunk_size": 800, "k": 8, "rerank": True},
]

print(experiments)
# Run the same evaluation set for each configuration and compare metrics.

## Evaluation

Evaluate each retrieval configuration using Recall@K/MRR plus answer metrics. Keep the test set fixed while changing one factor at a time. Record configuration + scores.

## Improvement ideas

Use a failure taxonomy: miss = recall problem; noisy top-k = precision/ranking problem; answer unsupported = grounding problem; prompt too large = context problem.

## Practice exercises

1. Add BM25 or keyword search.
2. Add a cross-encoder reranker.
3. Compare baseline RAG against advanced RAG on 20 queries.

## Advanced RAG concept map

Advanced RAG solves different failure modes at different stages:

| Problem | Technique | Goal |
|---|---|---|
| Vague query | Query rewriting | Better search query |
| One wording misses evidence | Multi-query | Search multiple formulations |
| Complex question | Query decomposition | Break into sub-questions |
| Query and document wording differ | HyDE | Search from a hypothetical answer |
| Exact names/IDs matter | Hybrid search | Combine lexical + semantic signals |
| Similar chunks dominate | MMR | Improve diversity |
| Vector ranking is noisy | Reranking | Better relevance order |
| Correct evidence is too large/noisy | Context compression | Smaller useful context |
| Small retrieval unit lacks context | Parent-child retrieval | Retrieve small, return larger context |

In [ ]:
# Educational query-decomposition example

def decompose_question(question):
    # A production version can ask an LLM to generate sub-questions.
    if 'compare' in question.lower() and 'and' in question.lower():
        parts = [p.strip() for p in question.split(' and ', 1)]
        return [f'What is {parts[0]}?', f'What is {parts[1]}?', question]
    return [question]

print(decompose_question('Compare FAISS and Chroma'))

In [ ]:
# Multi-query merge: keep the best evidence from several searches.
def deduplicate_results(result_lists, id_key='id'):
    seen = {}
    for results in result_lists:
        for item in results:
            seen[item[id_key]] = item
    return list(seen.values())

queries = [
    ['d1', 'd2', 'd3'],
    ['d2', 'd4'],
    ['d3', 'd5'],
]
print(deduplicate_results(
    [[{'id': x} for x in group] for group in queries]
))

In [ ]:
# Reciprocal Rank Fusion (RRF) is a useful hybrid/ranking baseline.
def rrf(rankings, k=60):
    scores = {}
    for ranking in rankings:
        for rank, doc_id in enumerate(ranking, start=1):
            scores[doc_id] = scores.get(doc_id, 0) + 1 / (k + rank)
    return sorted(scores.items(), key=lambda x: x[1], reverse=True)

print(rrf([['a', 'b', 'c'], ['b', 'c', 'd']]))

In [ ]:
# Parent-child retrieval data model
parent = {
    'id': 'p1',
    'text': 'Large section containing several related ideas.',
}
children = [
    {'id': 'c1', 'parent_id': 'p1', 'text': 'Small retrievable unit A.'},
    {'id': 'c2', 'parent_id': 'p1', 'text': 'Small retrievable unit B.'},
]

# Search children, then map the hit back to its parent.
hit = children[0]
print('retrieved child:', hit)
print('return parent:', parent)

## Advanced RAG evaluation

Do not evaluate every technique using the final answer only. Measure the stage you changed:

1. query transformation → did the rewritten query improve retrieval recall?
2. hybrid retrieval → did Recall@K / MRR improve?
3. reranking → did relevant chunks move upward?
4. compression → did faithfulness stay stable while context size fell?
5. parent-child → did answer quality improve without losing retrieval precision?

Keep the same benchmark while changing one component at a time. This turns Advanced RAG from a bag of tricks into an engineering process.

## Final checklist

- [ ] I can explain the concept in simple words.
- [ ] I can describe where it fits in a GenAI system.
- [ ] I can run and modify the code.
- [ ] I know at least one way to evaluate it.
- [ ] I know what to change when it fails.

## Research / reference notes


- LangChain Python reference: https://reference.langchain.com/python/langchain/overview
- LangGraph Python reference: https://reference.langchain.com/python/langgraph/overview
- Ragas metrics: https://docs.ragas.io/en/latest/concepts/metrics/available_metrics/
- OpenAI function calling guide: https://developers.openai.com/api/docs/guides/function-calling
- OpenAI Responses API: https://developers.openai.com/api/reference/cli/resources/responses/methods/create